# AOP-ProCon — Step 6: Seed spread, ESM-2 scale sweep and reverse fold direction

Step 5 reported single-seed, single-direction results. Three claims in it rest on that and are re-examined here:

| Claim in Step 5 | What Step 6 does |
|---|---|
| "Trained contrastive models are worse than the untrained ESM-2 / composition / length baselines." | Re-trains Primary, HP1-Random, k-NN-Pairing, Primary-MaxSpread and MechEqual-MaxSpread with **several seeds** and reports mean ± SD, range and the **number of seeds** that fall below the untrained baseline. |
| "Real pairs and random pairs are indistinguishable (HP1)." | Per-seed paired difference Primary − HP1-Random (same seed, same queries): mean ± SD, range, sign count. |
| "Scale does not help (HP5)." | Trains the same recipe on **35M / 150M / 650M / 3B** embeddings (several seeds each) and compares every trained head with the **untrained embedding of the same scale** on the same queries. |
| "Results hold in the direction train fold 0 → evaluate fold 1." | **Reverse direction**: pairs rebuilt with fold 1 as the training fold (scaler, threshold, pairs, random control — same code path as Step 3, verified to reproduce the Step 3 files when run with fold 0), evaluate on fold 0. |

### How the code is kept identical to Steps 3–5
* The training functions (`make_batches`, losses, `Prototypes`, `train_model`) are the **verbatim code of Step 4**, with one change: the order of the training batches also depends on the seed (Step 4 used the epoch number only, so seeds differed only in the head initialisation). With seed 42 the batch order is unchanged, so **seed 42 must reproduce the Step 4 checkpoint** — this is checked.
* Evaluation uses the Step 5 functions (MC retrieval, per-query AP, exact/permutation chance, bootstrap CI) on the same `strict` query / gallery sets.
* Provenance gates are the Step 5 ones (S2 fingerprint, embedding / descriptor / 12 pair-file SHA-256).

### Cost
Roughly 30–40 s per model on a Colab CPU (the 3B scale is slower). The defaults below (5 seeds for the main comparison, 3 for the scale sweep, 3 for the reverse direction) are 25 + 24 + 9 = 58 models, i.e. about **40–70 minutes**. Lower `N_SEEDS*` (a few cells below) for a quick run; the conclusions are generated from whatever was run.

### Limits
Seeds change the head initialisation and the training batch order, not the data split. Everything is still one S2, one pairing rule and one loss. Four scales are too few for a correlation coefficient, so none is computed.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"

In [ ]:
import os, json, hashlib, itertools, math, datetime
from importlib import metadata
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import wilcoxon
from scipy.special import comb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

PROC = f"{BASE}/data/processed"; EMB = f"{BASE}/data/embeddings"; RES = f"{BASE}/results"
os.makedirs(RES, exist_ok=True)
device = "cpu"; SEED = 42; N_BOOT = 2000
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": _ver("scikit-learn"), "scipy": _ver("scipy")}
print(VERSIONS)

def s2_fingerprint(s2_df):
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

In [ ]:
import copy, collections, time
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import pdist, squareform

N_SEEDS         = 5      # main comparison (650M), seeds 42, 43, ...
N_SEEDS_SCALE   = 3      # scale sweep
N_SEEDS_REVERSE = 3      # reverse fold direction
SEEDS = [SEED + i for i in range(N_SEEDS)]
T_START = time.time()

---
## 0. Load, provenance gates, evaluation sets

Everything must come from the same S2 / Step 2 / Step 3 / Step 4 run: S2 fingerprint, embedding and descriptor SHA-256 (Step 2 manifest), pair-file SHA-256 (Step 3 manifest, stamped into every checkpoint by Step 4). A mismatch stops the notebook.

In [ ]:
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"
aop = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")
desc = pd.read_csv(f"{PROC}/descriptors.csv")
need = {"sequence", "labels", "sample_weight", "lso_fold", "cluster_40", "homology_group", "strict_evidence", "homology_resolvable"}
assert need <= set(aop.columns) and aop["sequence"].is_unique and desc["sequence"].is_unique
assert set(aop["sequence"]) == set(desc["sequence"])
desc = desc.set_index("sequence").loc[aop["sequence"]].reset_index()
S2_FP = s2_fingerprint(aop)
m2 = json.load(open(f"{PROC}/step2_manifest.json")); pm = json.load(open(f"{PROC}/pairs_manifest.json"))
assert S2_FP == m2["s2_fingerprint"] == pm["s2_fingerprint"], "S2 / Step 2 / Step 3 fingerprints disagree"
assert S2_FP == EXPECTED_S2_FINGERPRINT, f"S2 fingerprint {S2_FP} is not the validated Step 1 build v3"
assert pm.get("build", "").startswith("v4")
assert sha256_file(f"{PROC}/descriptors.csv") == m2["sha256"]["descriptors.csv"]
MODES, PRIMARY_MODE = pm["modes"], pm["primary_mode"]
for key, h in pm["sha256"].items():
    assert sha256_file(f"{PROC}/pairs/{key}") == h, f"pairs/{key} differs from the file Step 3 recorded"
print("S2", S2_FP, "| descriptors + 12 pair files agree with the manifests | modes:", list(MODES), "| primary:", PRIMARY_MODE)

SCALES = {"35M": 480, "150M": 640, "650M": 1280, "3B": 2560}
ESM = {}
for sc, dim in SCALES.items():
    a = np.load(f"{EMB}/esm2_{sc}.npy")
    assert sha256_file(f"{EMB}/esm2_{sc}.npy") == m2["sha256"][f"esm2_{sc}.npy"], f"esm2_{sc}.npy differs from Step 2's record"
    assert a.shape == (len(aop), dim) and np.isfinite(a).all()
    assert open(f"{EMB}/esm2_{sc}_sequences.txt").read().splitlines() == aop["sequence"].tolist()
    ESM[sc] = dict(zip(aop["sequence"], a))
AA = "ACDEFGHIKLMNPQRSTVWY"
AAC = {s: np.array([s.count(a) / len(s) for a in AA]) for s in aop["sequence"]}
DESC = dict(zip(desc["sequence"], desc[[c for c in desc.columns if c != "sequence"]].values))
DCOLS = [c for c in desc.columns if c != "sequence"]
LENGTH = {s: len(s) for s in aop["sequence"]}

mech = aop[aop["labels"].str.contains("FRS|MC")].reset_index(drop=True)
assert mech["lso_fold"].notna().all() and mech["homology_group"].notna().all()
for col in ("sequence", "cluster_40", "homology_group"):
    assert mech.groupby(col)["lso_fold"].nunique().max() == 1, f"{col} spans both folds"
gen_seqs = aop[aop["labels"] == "GEN"]["sequence"].tolist()

def mode_pool(mode):
    d = mech
    if MODES[mode]["strict"]: d = d[d["strict_evidence"].astype(bool)]
    if MODES[mode]["ge4"]:    d = d[d["homology_resolvable"].astype(bool)]
    return d.reset_index(drop=True)
POOL = {m: mode_pool(m) for m in MODES}
EVAL = {}
for m, d in POOL.items():
    q, g = d[d.lso_fold == 1].reset_index(drop=True), d[d.lso_fold == 0].reset_index(drop=True)
    for col in ("sequence", "cluster_40", "homology_group"):
        assert not (set(q[col]) & set(g[col])), f"{m}: query/gallery share {col}"
    EVAL[m] = {"query": q, "gallery": g}
rows = []
for m, e in EVAL.items():
    for part in ("query", "gallery"):
        vc = e[part]["labels"].value_counts()
        rows.append({"mode": m, "set": part, "n": len(e[part]), "FRS": int(vc.get("FRS", 0)), "FRS;MC": int(vc.get("FRS;MC", 0)), "MC": int(vc.get("MC", 0))})
print(pd.DataFrame(rows).to_string(index=False))

### Evaluation functions (identical to Step 5)

In [ ]:
def unit(a): return a / np.linalg.norm(a, axis=1, keepdims=True)

def base_sims(rep, q_df, g_df, rng):
    qs, gs = q_df["sequence"].tolist(), g_df["sequence"].tolist()
    if rep == "RAW 650M": return unit(np.stack([ESM["650M"][s] for s in qs])) @ unit(np.stack([ESM["650M"][s] for s in gs])).T
    if rep == "RAW 150M": return unit(np.stack([ESM["150M"][s] for s in qs])) @ unit(np.stack([ESM["150M"][s] for s in gs])).T
    if rep == "AAC-20":   return unit(np.stack([AAC[s] for s in qs])) @ unit(np.stack([AAC[s] for s in gs])).T
    if rep == "DESC-14":
        sc = StandardScaler().fit(np.stack([DESC[s] for s in gs]))
        return unit(sc.transform(np.stack([DESC[s] for s in qs]))) @ unit(sc.transform(np.stack([DESC[s] for s in gs]))).T
    if rep == "LENGTH":
        lq, lg = np.array([LENGTH[s] for s in qs]), np.array([LENGTH[s] for s in gs])
        return -np.abs(lq[:, None] - lg[None, :]).astype(float) + 1e-6 * rng.rand(len(qs), len(gs))      # tiny jitter: ties broken at random
    raise KeyError(rep)

BASELINES = ["RAW 650M", "RAW 150M", "AAC-20", "DESC-14", "LENGTH"]

def per_query(S, q_df, g_df, target):
    rel_g = g_df["labels"].str.contains(target).values.astype(int)
    qi = np.where(q_df["labels"].str.contains(target).values)[0]
    rows = []
    for i in qi:
        s = S[i]; order = np.argsort(-s, kind="stable"); r = rel_g[order]
        rows.append({"sequence": q_df["sequence"].iloc[i], "AP": average_precision_score(rel_g, s),
                     "R@5": float(r[:5].any()), "R@10": float(r[:10].any())})
    return pd.DataFrame(rows).set_index("sequence")

def chance(g_df, target, n_perm=500, seed=0):
    rel = g_df["labels"].str.contains(target).values.astype(int); N, R = len(rel), int(rel.sum())
    rec = {k: (1.0 if N - R < k else float(1 - comb(N - R, k, exact=True) / comb(N, k, exact=True))) for k in (5, 10)}
    rng = np.random.RandomState(seed)
    ap = float(np.mean([average_precision_score(rel, rng.rand(N)) for _ in range(n_perm)]))
    return {"N_gallery": N, "relevant": R, "chance_R@5": rec[5], "chance_R@10": rec[10], "chance_AP": ap}

def boot_mean_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float); rng = np.random.RandomState(seed)
    b = x[rng.randint(0, len(x), (n, len(x)))].mean(1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def spread(E):
    z = unit(E); n = len(z)
    mean_cos = float(((z @ z.T).sum() - n) / (n * (n - 1)))
    s2 = np.linalg.svd(z - z.mean(0, keepdims=True), compute_uv=False) ** 2
    p = s2[s2 > 0] / s2.sum()
    return mean_cos, float(np.exp(-(p * np.log(p)).sum()))

CHANCE = {(m, t): chance(EVAL[m]["gallery"], t) for m in MODES for t in ("MC", "FRS")}
print(pd.DataFrame([{"mode": m, "task": t, **v, "carries_information": v["chance_R@5"] < 0.99} for (m, t), v in CHANCE.items()]).round(4).to_string(index=False))
print("\nThe FRS task is saturated in every mode (nearly every gallery peptide is FRS-active); only MC retrieval can separate representations.")

In [ ]:
P = PRIMARY_MODE
seq_to_weight = dict(zip(aop["sequence"], aop["sample_weight"]))
PAIRS = {n: pd.read_csv(f"{PROC}/pairs/{P}/{n}.csv") for n in ("positive_pairs", "knn_pairs_k8", "ablation_random_pairs")}
Q, G = EVAL[P]["query"], EVAL[P]["gallery"]
PRIMARY_POOL = G                                   # fold-0 strict mechanism-active peptides = Step 4's prototype / ARI pool
CONFIG = dict(tau=0.2, lam=5.0, lr=1e-4, use_uniformity=True, uniformity_weight=1.5, lr_decay=True, grad_clip=1.0)   # identical to Step 4
def split_by_fold(df): return df[df.fold == 0].reset_index(drop=True), df[df.fold == 1].reset_index(drop=True)
TR = {n: split_by_fold(PAIRS[n]) for n in PAIRS}
print({n: (len(a), len(b)) for n, (a, b) in TR.items()}, "| query", len(Q), "gallery", len(G))
print("MC-active queries:", int(Q["labels"].str.contains("MC").sum()), "| chance:", {k: round(v, 3) for k, v in CHANCE[(P, "MC")].items()})

---
## 1. Training code — verbatim from Step 4

(`make_batches`, `positive_only_loss`, `uniformity_loss`, `Prototypes`, `proto_distances`, `train_model`; the only edit is that the training-batch order depends on the seed, see the title cell.)

In [ ]:
def make_batches(df, batch_size=64, seed=None, mc_mode="natural"):
    rng = np.random.RandomState(seed); batches = []
    frs = df[df["mechanism"] == "FRS"].sample(frac=1, random_state=seed).reset_index(drop=True)
    for start in range(0, len(frs), batch_size):
        chunk = frs.iloc[start:start + batch_size]
        if len(chunk) > 0: batches.append((chunk, "FRS"))
    n_frs = len(batches)
    mc = df[df["mechanism"] == "MC"].reset_index(drop=True)
    if mc_mode == "equal" and len(mc) > 0 and n_frs > 0:
        # mechanism-equal steps: as many MC batches per epoch as FRS batches, MC rows resampled (inverse anchor-frequency weights)
        freq = collections.Counter(list(mc["seq1"]) + list(mc["seq2"]))
        w = np.array([1.0 / (freq[r.seq1] + freq[r.seq2]) for r in mc.itertuples()]); w = w / w.sum()
        idx_all = rng.choice(len(mc), size=n_frs * batch_size, replace=True, p=w)
        for start in range(0, len(idx_all), batch_size):
            batches.append((mc.iloc[idx_all[start:start + batch_size]], "MC"))
    elif mc_mode == "balanced" and len(mc) > 0 and n_frs > 0:
        freq = collections.Counter(list(mc["seq1"]) + list(mc["seq2"]))
        w = np.array([1.0 / (freq[r.seq1] + freq[r.seq2]) for r in mc.itertuples()]); w = w / w.sum()
        idx_all = rng.choice(len(mc), size=len(mc), replace=True, p=w)
        for start in range(0, len(idx_all), batch_size):
            idx = idx_all[start:start + batch_size]
            if len(idx) > 0: batches.append((mc.iloc[idx], "MC"))
    else:
        for start in range(0, len(mc), batch_size):
            chunk = mc.iloc[start:start + batch_size]
            if len(chunk) > 0: batches.append((chunk, "MC"))
    rng.shuffle(batches); return batches

def positive_only_loss(anchor, positive, weights=None, tau=0.07):
    anchor = F.normalize(anchor, dim=-1); positive = F.normalize(positive, dim=-1)
    sim = (anchor * positive).sum(-1) / tau
    if weights is None: return -sim.mean()
    w = weights / weights.sum(); return -(sim * w).sum()

def uniformity_loss(x):
    # Wang & Isola (2020). x: normalized [N,D]. Minimising it spreads points on the hypersphere.
    if x.shape[0] < 2: return torch.tensor(0.0, device=x.device)
    sq = torch.pdist(x, p=2).pow(2)
    return sq.mul(-2.0).exp().mean().log()

class Prototypes:
    def __init__(self, init_vectors, dynamic, ema=0.99):
        self.vectors = {k: F.normalize(v, dim=-1) for k, v in init_vectors.items()}
        self.dynamic = dynamic; self.ema = ema
    def update(self, mech, batch_mean):
        if mech not in self.dynamic: return
        self.vectors[mech] = F.normalize(self.ema * self.vectors[mech] + (1 - self.ema) * batch_mean, dim=-1)
    def push_away_loss(self, emb, mech, lam=5.0):
        other = [v for k, v in self.vectors.items() if k != mech]
        sim = sum(F.cosine_similarity(emb, o.unsqueeze(0).expand_as(emb)).mean() for o in other)
        return lam * sim
    def pairwise_distance_report(self):
        ks = list(self.vectors.keys())
        return {f"{ks[i]}-{ks[j]}": (1 - (self.vectors[ks[i]] * self.vectors[ks[j]]).sum()).item()
                for i in range(len(ks)) for j in range(i + 1, len(ks))}

def proto_distances(d):
    ks = sorted(d.keys())
    return {f"{ks[i]}-{ks[j]}": float(1 - (F.normalize(d[ks[i]], dim=-1) *
                                           F.normalize(d[ks[j]], dim=-1)).sum())
            for i in range(len(ks)) for j in range(i + 1, len(ks))}

In [ ]:
def train_model(train_pairs, val_pairs, embed_lookup, input_dim, run_name,
                mc_mode="natural", checkpoint_mode="val_loss",
                tau=0.07, lam=5.0, lr=1e-3, use_uniformity=False, uniformity_weight=1.0,
                lr_decay=False, grad_clip=None,
                max_epochs=40, patience=12, checkpoint_threshold=0.5,
                saved_dist_floor=None, seed=SEED, verbose=True, proto_pool=None):
    torch.manual_seed(seed); np.random.seed(seed)          # per-model reseed: models stay comparable
    pool = PRIMARY_POOL if proto_pool is None else proto_pool            # fold-0 peptides of the mode: prototype initialisation and ARI
    head = torch.nn.Linear(input_dim, 128).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
                 if lr_decay else None)

    def embed_batch(chunk, col):
        return torch.tensor(np.stack([embed_lookup[s] for s in chunk[col]]),
                            dtype=torch.float32).to(device)
    def batch_weights(chunk):
        return torch.tensor([(seq_to_weight[a] + seq_to_weight[b]) / 2.0
                             for a, b in zip(chunk["seq1"], chunk["seq2"])],
                            dtype=torch.float32).to(device)
    def project(seqs):
        raw = torch.tensor(np.stack([embed_lookup[s] for s in seqs]), dtype=torch.float32).to(device)
        with torch.no_grad(): out = head(raw)
        return F.normalize(out, dim=-1)
    def epoch_ari():
        emb = project(pool["sequence"].tolist()).numpy()
        lab = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(emb)
        return float(adjusted_rand_score(pool["labels"].values, lab))

    def spread_stats():
        # collapse diagnostics on the mode's fold-0 mechanism-active peptides (no threshold is applied to them):
        #   mean_cos : mean pairwise cosine of the unit-norm embeddings (1.0 = every peptide at one point)
        #   eff_rank : exp(entropy of the normalised squared singular values of the centred embeddings) = number of directions in use
        z = project(pool["sequence"].tolist())
        mean_cos = float(((z @ z.T).sum() - len(z)) / (len(z) * (len(z) - 1)))
        s2 = torch.linalg.svdvals(z - z.mean(0, keepdim=True)).numpy() ** 2
        p = s2[s2 > 0] / s2.sum()
        return mean_cos, float(np.exp(-(p * np.log(p)).sum()))

    gen_raw = torch.tensor(np.stack([embed_lookup[s] for s in gen_seqs]),
                           dtype=torch.float32).to(device)
    def gen_trained_for(head_state):            # N-10: the GEN vector that WOULD be saved
        probe = torch.nn.Linear(input_dim, 128).to(device)
        probe.load_state_dict(head_state)
        with torch.no_grad():
            return F.normalize(probe(gen_raw).mean(0), dim=-1)
    def mech_centroid(mech):
        seqs = (pool[pool["labels"].str.contains(mech)]["sequence"].unique().tolist())
        raw = torch.tensor(np.stack([embed_lookup[s] for s in seqs]), dtype=torch.float32).to(device)
        with torch.no_grad(): return head(raw).mean(0)

    with torch.no_grad(): gen_init = head(gen_raw).mean(0)
    prototypes = Prototypes({"FRS": mech_centroid("FRS"), "MC": mech_centroid("MC"), "GEN": gen_init},
                            dynamic={"FRS", "MC"}, ema=0.99)

    def run_batch(chunk, mech, train=True):
        a_raw = embed_batch(chunk, "seq1"); p_raw = embed_batch(chunk, "seq2")
        if train:
            a = head(a_raw); p = head(p_raw)
        else:
            with torch.no_grad(): a = head(a_raw); p = head(p_raw)
        loss = (positive_only_loss(a, p, weights=batch_weights(chunk), tau=tau)
                + prototypes.push_away_loss(a, mech, lam=lam))
        if use_uniformity:
            loss = loss + uniformity_weight * uniformity_loss(F.normalize(a, dim=-1))
        return loss, a

    best_state, best_train_safe, best_ari, since = None, float("inf"), -1.0, 0
    best_cos = float("inf")
    fallback_state, fallback_dist = None, -1.0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "proto_dist": [],
               "proto_dist_saved": [], "ari": [], "mean_cos": [], "eff_rank": []}

    for epoch in range(max_epochs):
        head.train(); tl = []
        for chunk, mech in make_batches(train_pairs, 64, seed=epoch + 10000 * (seed - SEED), mc_mode=mc_mode):
            loss, a = run_batch(chunk, mech, True)
            optimizer.zero_grad(); loss.backward()
            if grad_clip is not None: torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)
            optimizer.step()
            prototypes.update(mech, a.mean(0).detach()); tl.append(loss.item())
        head.eval(); vl = []
        for chunk, mech in make_batches(val_pairs, 64, seed=1000 + epoch, mc_mode="natural"):
            loss, _ = run_batch(chunk, mech, False); vl.append(loss.item())
        train_loss = float(np.mean(tl)); val_loss = float(np.mean(vl)) if vl else float("nan")
        if scheduler is not None: scheduler.step()

        min_dist = min(prototypes.pairwise_distance_report().values())        # old guard value
        cand = {k: v.clone() for k, v in prototypes.vectors.items()}          # N-10: what gets saved
        cand["GEN"] = gen_trained_for(head.state_dict())
        saved_d = proto_distances(cand)
        min_dist_saved = min(saved_d.values())
        mean_cos, eff_rank = spread_stats()

        cur_ari = epoch_ari() if checkpoint_mode == "ari" else float("nan")
        for k, v in zip(["epoch", "train_loss", "val_loss", "proto_dist", "proto_dist_saved", "ari", "mean_cos", "eff_rank"],
                        [epoch, train_loss, val_loss, min_dist, min_dist_saved, cur_ari, mean_cos, eff_rank]):
            history[k].append(v)

        snap = lambda: {"head": copy.deepcopy(head.state_dict()),
                        "prototypes": {k: v.clone() for k, v in prototypes.vectors.items()},
                        "prototypes_saved": {k: v.clone() for k, v in cand.items()},
                        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "min_proto_dist": min_dist, "min_proto_dist_saved": min_dist_saved,
                        "saved_distances": saved_d, "ari": cur_ari, "mean_cos": mean_cos, "eff_rank": eff_rank}

        if min_dist_saved > fallback_dist:          # best-separated epoch, criterion aside
            fallback_dist, fallback_state = min_dist_saved, snap()

        improved = False
        if checkpoint_mode == "ari":
            # E-013: optional floor on the SAVED prototypes, so ARI cannot select a collapsed state
            floor_ok = (saved_dist_floor is None) or (min_dist_saved > saved_dist_floor)
            if floor_ok and cur_ari > best_ari:
                best_ari = cur_ari; best_state = snap(); improved = True
        elif checkpoint_mode == "spread":
            # keep the epoch whose embeddings are the most spread out (lowest mean pairwise cosine); no label and no loss involved
            if mean_cos < best_cos:
                best_cos = mean_cos; best_state = snap(); improved = True
        else:
            if min_dist > checkpoint_threshold and train_loss < best_train_safe:
                best_train_safe = train_loss; best_state = snap(); improved = True

        if np.isnan(train_loss):
            if verbose: print(f"  [{run_name}] Epoch {epoch}: NaN -- stopping.")
            break
        flag = " COLLAPSE" if min_dist < 0.1 else ""
        flag += " GEN-MERGED" if min_dist_saved < 0.1 <= min_dist else ""
        if verbose and (epoch % 5 == 0 or epoch < 6 or flag):
            print(f"  [{run_name}] Epoch {epoch:3d} | train={train_loss:.4f} | val={val_loss:.4f} "
                  f"| guard={min_dist:.4f} | saved={min_dist_saved:.4f} | mean_cos={mean_cos:.3f} eff_rank={eff_rank:.1f}{flag}")
        if improved: since = 0
        elif best_state is not None:
            since += 1
            if since >= patience:
                if verbose: print(f"  [{run_name}] Early stopping at epoch {epoch}")
                break

    criterion_met = best_state is not None
    if not criterion_met and fallback_state is not None:
        # Never abandon a whole 10-model run because one model failed the guard: keep the
        # epoch with the best SAVED prototype separation and flag it loudly instead.
        best_state = fallback_state
        if verbose:
            print(f"  [{run_name}] *** NO epoch met the selection criterion. Falling back to the "
                  f"best-separated epoch ({best_state['epoch']}, saved min_dist="
                  f"{best_state['min_proto_dist_saved']:.4f}). This model is NOT comparable to "
                  f"the others -- report it as 'criterion not met'.")
    if best_state is not None:
        best_state["criterion_met"] = criterion_met
        head.load_state_dict(best_state["head"]); prototypes.vectors = best_state["prototypes"]
        if verbose:
            print(f"  [{run_name}] Restored epoch {best_state['epoch']}: "
                  f"guard min_dist={best_state['min_proto_dist']:.4f}, "
                  f"SAVED min_dist={best_state['min_proto_dist_saved']:.4f}")
            if best_state["min_proto_dist_saved"] < 0.1:
                print(f"  [{run_name}] *** WARNING (N-10): saved prototypes are merged "
                      f"({best_state['saved_distances']}). The guard value above does NOT "
                      f"describe this checkpoint.")
    elif verbose:
        print(f"  [{run_name}] WARNING: no checkpoint at all (training produced nothing usable).")
    return {"head": head, "prototypes": prototypes, "history": history,
            "best_state": best_state, "embed_lookup": embed_lookup,
            "run_name": run_name, "input_dim": input_dim}

In [ ]:
def head_embed(head, lookup, seqs):
    raw = torch.tensor(np.stack([lookup[s] for s in seqs]), dtype=torch.float32)
    with torch.no_grad(): return F.normalize(head(raw), dim=-1).numpy()

def eval_run(run, lookup, q_df, g_df, raw_ap=None):
    head = run["head"]; head.eval()
    Eq, Eg = head_embed(head, lookup, q_df["sequence"].tolist()), head_embed(head, lookup, g_df["sequence"].tolist())
    pq = per_query(Eq @ Eg.T, q_df, g_df, "MC"); mc, er = spread(Eq); bs = run["best_state"]
    rec = {"AP": pq["AP"].mean(), "R@5": pq["R@5"].mean(), "R@10": pq["R@10"].mean(), "mean_cos_heldout": mc, "eff_rank_heldout": er,
           "epoch": bs["epoch"], "criterion_met": bool(bs.get("criterion_met", True))}
    if raw_ap is not None:
        rec["dAP_vs_raw"] = float((pq["AP"] - raw_ap).mean())
    return rec, pq

def raw_per_query(scale, q_df, g_df):
    S = unit(np.stack([ESM[scale][s] for s in q_df["sequence"]])) @ unit(np.stack([ESM[scale][s] for s in g_df["sequence"]])).T
    return per_query(S, q_df, g_df, "MC")

---
## 2. Seed spread (650M, `strict` pairs, train fold 0 → evaluate fold 1)

Five training recipes, each with several seeds. Every trained head is compared **on the same queries** with the untrained ESM-2 650M embedding (`dAP_vs_raw` = mean over queries of AP(trained) − AP(raw)). The seed-42 Primary head is first checked against the Step 4 checkpoint.

In [ ]:
CFGS = {
    "Primary":             dict(train="positive_pairs",        mc_mode="natural", ckpt="val_loss"),
    "HP1-Random":          dict(train="ablation_random_pairs", mc_mode="natural", ckpt="val_loss"),
    "k-NN-Pairing":        dict(train="knn_pairs_k8",          mc_mode="natural", ckpt="val_loss"),
    "Primary-MaxSpread":   dict(train="positive_pairs",        mc_mode="natural", ckpt="spread"),
    "MechEqual-MaxSpread": dict(train="positive_pairs",        mc_mode="equal",   ckpt="spread"),
}
def run_cfg(name, cfg, seed, lookup, dim, train_pairs=None, val_pairs=None, pool=None):
    tr, va = TR[cfg["train"]] if train_pairs is None else (train_pairs, val_pairs)
    extra = dict(patience=40) if cfg["ckpt"] == "spread" else {}
    return train_model(tr, va, lookup, dim, f"{name}/s{seed}", mc_mode=cfg["mc_mode"], checkpoint_mode=cfg["ckpt"], seed=seed,
                       verbose=False, proto_pool=(PRIMARY_POOL if pool is None else pool), **CONFIG, **extra)

RAW650 = raw_per_query("650M", Q, G); raw_ap = RAW650["AP"]
print(f"untrained ESM-2 650M: AP={raw_ap.mean():.4f} | chance AP={CHANCE[(P, 'MC')]['chance_AP']:.4f}")

rows = []
for seed in SEEDS:
    for name, cfg in CFGS.items():
        run = run_cfg(name, cfg, seed, ESM["650M"], 1280)
        rec, _ = eval_run(run, ESM["650M"], Q, G, raw_ap)
        rows.append({"model": name, "seed": seed, **rec})
        if name == "Primary" and seed == SEED:
            ref = torch.load(f"{RES}/model_primary.pt", map_location="cpu", weights_only=False)["head_state_dict"]
            diff = max(float((run["head"].state_dict()[k] - ref[k]).abs().max()) for k in ref)
            print(f"reproduction check: seed {SEED} Primary head vs Step 4 checkpoint, max |diff| = {diff:.2e} -> {'REPRODUCES Step 4' if diff < 1e-5 else 'DOES NOT reproduce Step 4 (different hardware / library versions?)'}")
    print(f"seed {seed} done  (t = {time.time() - T_START:.0f}s)", flush=True)
SEEDDF = pd.DataFrame(rows); SEEDDF.to_csv(f"{RES}/step6_seed_runs.csv", index=False)

agg = SEEDDF.groupby("model").agg(n=("seed", "count"), AP_mean=("AP", "mean"), AP_sd=("AP", "std"), AP_min=("AP", "min"), AP_max=("AP", "max"),
                                  dAP_mean=("dAP_vs_raw", "mean"), dAP_sd=("dAP_vs_raw", "std"), dAP_max=("dAP_vs_raw", "max"),
                                  mean_cos=("mean_cos_heldout", "mean"), eff_rank=("eff_rank_heldout", "mean")).reset_index()
agg["seeds_below_raw"] = [int((SEEDDF[SEEDDF.model == m]["dAP_vs_raw"] < 0).sum()) for m in agg["model"]]
agg["raw_AP"] = raw_ap.mean(); agg["chance_AP"] = CHANCE[(P, "MC")]["chance_AP"]
print("\n", agg.round(4).to_string(index=False)); agg.to_csv(f"{RES}/step6_seed_spread.csv", index=False)

pv = SEEDDF.pivot(index="seed", columns="model", values="AP")
d_hp1 = (pv["Primary"] - pv["HP1-Random"])
d_sel = (pv["Primary-MaxSpread"] - pv["Primary"])
print(f"\nHP1 per-seed paired difference AP(Primary) - AP(HP1-Random): mean {d_hp1.mean():+.4f}, SD {d_hp1.std():.4f}, range [{d_hp1.min():+.4f}, {d_hp1.max():+.4f}], "
      f"positive in {int((d_hp1 > 0).sum())}/{len(d_hp1)} seeds")
print(f"selection rule AP(MaxSpread) - AP(Primary): mean {d_sel.mean():+.4f}, SD {d_sel.std():.4f}, range [{d_sel.min():+.4f}, {d_sel.max():+.4f}], positive in {int((d_sel > 0).sum())}/{len(d_sel)} seeds")

---
## 3. ESM-2 scale sweep (contrastive side of HP5)

For each scale the **same recipe** (Primary and Primary-MaxSpread, `strict` pairs) is trained on that scale's embeddings and compared with the **untrained embedding of the same scale** on the same queries. The point is whether scaling helps the *trained* model relative to its own untrained baseline; the probe side of HP5 is in Step 5.

In [ ]:
SC_SEEDS = SEEDS[:N_SEEDS_SCALE]; srows = []; raw_by_scale = {}
for sc, dim in SCALES.items():
    rq = raw_per_query(sc, Q, G); raw_by_scale[sc] = rq["AP"]
    lo, hi = boot_mean_ci(rq["AP"].values)
    for name in ("Primary", "Primary-MaxSpread"):
        for seed in SC_SEEDS:
            run = run_cfg(name, CFGS[name], seed, ESM[sc], dim)
            rec, _ = eval_run(run, ESM[sc], Q, G, rq["AP"])
            srows.append({"scale": sc, "dim": dim, "model": name, "seed": seed, "raw_AP": rq["AP"].mean(), "raw_lo": lo, "raw_hi": hi, **rec})
    print(f"scale {sc} done  (t = {time.time() - T_START:.0f}s)", flush=True)
SCALEDF = pd.DataFrame(srows); SCALEDF.to_csv(f"{RES}/step6_scale_runs.csv", index=False)
sagg = SCALEDF.groupby(["scale", "model"]).agg(dim=("dim", "first"), raw_AP=("raw_AP", "first"), raw_lo=("raw_lo", "first"), raw_hi=("raw_hi", "first"),
                                               AP_mean=("AP", "mean"), AP_sd=("AP", "std"), dAP_mean=("dAP_vs_raw", "mean"), dAP_sd=("dAP_vs_raw", "std"),
                                               mean_cos=("mean_cos_heldout", "mean"), n=("seed", "count")).reset_index()
sagg["seeds_below_raw"] = [int((SCALEDF[(SCALEDF.scale == r.scale) & (SCALEDF.model == r.model)]["dAP_vs_raw"] < 0).sum()) for r in sagg.itertuples()]
sagg["order"] = sagg["scale"].map({s: i for i, s in enumerate(SCALES)}); sagg = sagg.sort_values(["model", "order"]).drop(columns="order")
print(sagg.round(4).to_string(index=False)); sagg.to_csv(f"{RES}/step6_scale_sweep.csv", index=False)
print("\nUntrained-embedding AP by scale (MC retrieval, with 95% CI over queries):")
for sc in SCALES:
    lo, hi = boot_mean_ci(raw_by_scale[sc].values); print(f"  ESM-2 {sc:<4} AP={raw_by_scale[sc].mean():.4f} [{lo:.4f}, {hi:.4f}]")
print(f"  chance AP = {CHANCE[(P, 'MC')]['chance_AP']:.4f}")

---
## 4. Reverse fold direction (train fold 1 → evaluate fold 0)

The pairs are rebuilt with fold 1 as the training fold using the **same procedure as Step 3** (per-mechanism scaler and 10th-percentile threshold fitted on the training fold, pairs inside a fold, `d > 0`, random control count-matched with the fixed seed). With fold 0 as the fit fold the code is verified to reproduce the Step 3 `strict` pair files exactly before it is used for fold 1.

Note the asymmetry of the `strict` split: **all 18 strict MC-only peptides are in fold 1.** In the original direction they are queries and never seen in training; in the reverse direction they are in the training fold and the MC-active queries are the 55 dual peptides of fold 0.

In [ ]:
PCT_, MECH_ = 10, ("FRS", "MC")
def build_pairs(fit_fold, pool_df):
    parts = []
    for m in MECH_:
        Pm = pool_df[pool_df["labels"].str.contains(m)].drop_duplicates("sequence").reset_index(drop=True)
        P0 = Pm[Pm.lso_fold == fit_fold]
        sc = StandardScaler().fit(np.stack([DESC[s] for s in P0["sequence"]]))
        D0 = squareform(pdist(sc.transform(np.stack([DESC[s] for s in P0["sequence"]]))))
        thr = float(np.percentile(D0[np.triu_indices(len(P0), k=1)], PCT_))
        for f in (0, 1):
            Pf = Pm[Pm.lso_fold == f].reset_index(drop=True)
            D = squareform(pdist(sc.transform(np.stack([DESC[s] for s in Pf["sequence"]])))); up = np.triu(np.ones_like(D), k=1) > 0
            i, j = np.where((D > 0) & (D < thr) & up)
            parts.append(pd.DataFrame({"seq1": Pf["sequence"].values[i], "seq2": Pf["sequence"].values[j], "mechanism": m, "distance": D[i, j], "fold": f}))
    pos = pd.concat(parts, ignore_index=True)
    rng = np.random.default_rng(SEED); rp = []
    for m in MECH_:
        Pm = pool_df[pool_df["labels"].str.contains(m)].drop_duplicates("sequence").reset_index(drop=True)
        for f in (0, 1):
            Pf = Pm[Pm.lso_fold == f].reset_index(drop=True); n_want = int(((pos.mechanism == m) & (pos.fold == f)).sum()); seen = set()
            while len(seen) < n_want:
                a, b = rng.integers(0, len(Pf), 2)
                if a != b: seen.add((int(min(a, b)), int(max(a, b))))
            seen = sorted(seen)
            rp.append(pd.DataFrame({"seq1": [Pf["sequence"].iloc[a] for a, _ in seen], "seq2": [Pf["sequence"].iloc[b] for _, b in seen], "mechanism": m, "distance": np.nan, "fold": f}))
    return pos, pd.concat(rp, ignore_index=True)

def key(df): return {(min(a, b), max(a, b), m, f) for a, b, m, f in zip(df.seq1, df.seq2, df.mechanism, df.fold)}
pos0, rnd0 = build_pairs(0, POOL[P])
assert key(pos0) == key(PAIRS["positive_pairs"]), "fold-0 reconstruction does not reproduce the Step 3 positive pairs"
assert key(rnd0) == key(PAIRS["ablation_random_pairs"]), "fold-0 reconstruction does not reproduce the Step 3 random control"
print(f"verification: pairs rebuilt with fold 0 as fit fold reproduce the Step 3 files exactly (positive {len(pos0)}, random {len(rnd0)})")

pos1, rnd1 = build_pairs(1, POOL[P])
pool1 = POOL[P][POOL[P].lso_fold == 1].reset_index(drop=True)          # reverse training pool (prototype / ARI)
Qr, Gr = POOL[P][POOL[P].lso_fold == 0].reset_index(drop=True), pool1
chr_ = chance(Gr, "MC")
print(f"reverse direction: train pairs FRS {int(((pos1.fold == 1) & (pos1.mechanism == 'FRS')).sum())} / MC {int(((pos1.fold == 1) & (pos1.mechanism == 'MC')).sum())}; "
      f"MC-active queries {int(Qr['labels'].str.contains('MC').sum())}; gallery MC-active {chr_['relevant']}/{chr_['N_gallery']}; chance AP={chr_['chance_AP']:.3f}, chance R@5={chr_['chance_R@5']:.3f}")

rev_sets = {"Primary": (pos1[pos1.fold == 1].reset_index(drop=True), pos1[pos1.fold == 0].reset_index(drop=True)),
            "HP1-Random": (rnd1[rnd1.fold == 1].reset_index(drop=True), rnd1[rnd1.fold == 0].reset_index(drop=True))}
rev_sets["Primary-MaxSpread"] = rev_sets["Primary"]
rng_ = np.random.RandomState(SEED)
rev_base = {"RAW 650M": per_query(base_sims("RAW 650M", Qr, Gr, rng_), Qr, Gr, "MC"), "AAC-20": per_query(base_sims("AAC-20", Qr, Gr, rng_), Qr, Gr, "MC"),
            "LENGTH": per_query(base_sims("LENGTH", Qr, Gr, rng_), Qr, Gr, "MC")}
for k, v in rev_base.items(): print(f"  untrained baseline {k:<9} AP={v['AP'].mean():.4f}  R@5={v['R@5'].mean():.3f}")
rrows = []
for seed in SEEDS[:N_SEEDS_REVERSE]:
    for name in ("Primary", "HP1-Random", "Primary-MaxSpread"):
        tr, va = rev_sets[name]
        run = run_cfg(name, CFGS[name], seed, ESM["650M"], 1280, train_pairs=tr, val_pairs=va, pool=pool1)
        rec, _ = eval_run(run, ESM["650M"], Qr, Gr, rev_base["RAW 650M"]["AP"]); rrows.append({"direction": "fold1->fold0", "model": name, "seed": seed, **rec})
    print(f"reverse seed {seed} done  (t = {time.time() - T_START:.0f}s)", flush=True)
REVDF = pd.DataFrame(rrows); REVDF.to_csv(f"{RES}/step6_reverse_runs.csv", index=False)
ragg = REVDF.groupby("model").agg(n=("seed", "count"), AP_mean=("AP", "mean"), AP_sd=("AP", "std"), dAP_mean=("dAP_vs_raw", "mean"), dAP_sd=("dAP_vs_raw", "std"),
                                  mean_cos=("mean_cos_heldout", "mean")).reset_index()
ragg["seeds_below_raw"] = [int((REVDF[REVDF.model == m]["dAP_vs_raw"] < 0).sum()) for m in ragg["model"]]
ragg["raw_AP"] = rev_base["RAW 650M"]["AP"].mean(); ragg["chance_AP"] = chr_["chance_AP"]
print("\n", ragg.round(4).to_string(index=False)); ragg.to_csv(f"{RES}/step6_reverse.csv", index=False)

---
## 5. Conclusions — generated from the tables above

In [ ]:
print("=" * 80); print("STEP 6 CONCLUSIONS (generated)"); print("=" * 80)
a = agg.set_index("model")
print(f"\n1. SEED SPREAD (650M, strict, fold 0 -> fold 1, {N_SEEDS} seeds). Untrained ESM-2 650M AP = {raw_ap.mean():.3f}; chance AP = {CHANCE[(P, 'MC')]['chance_AP']:.3f}.")
for m, r in a.iterrows():
    print(f"   {m:<22} AP {r['AP_mean']:.3f} +/- {r['AP_sd']:.3f} (range {r['AP_min']:.3f}-{r['AP_max']:.3f}); dAP vs untrained {r['dAP_mean']:+.3f} +/- {r['dAP_sd']:.3f}; "
          f"below the untrained baseline in {int(r['seeds_below_raw'])}/{int(r['n'])} seeds")
n_above = int((SEEDDF["dAP_vs_raw"] > 0).sum())
print(f"   -> {n_above} of {len(SEEDDF)} trained runs have a positive mean dAP over the untrained embedding"
      f"{'; the Step 5 conclusion (trained < untrained) is stable across seeds.' if n_above == 0 else '; the Step 5 conclusion is NOT uniformly stable across seeds.'}")
print(f"2. HP1 across seeds: AP(Primary) - AP(HP1-Random) = {d_hp1.mean():+.4f} +/- {d_hp1.std():.4f}, positive in {int((d_hp1 > 0).sum())}/{len(d_hp1)} seeds.")
print(f"   Seed-to-seed SD of Primary AP = {a.loc['Primary', 'AP_sd']:.4f}: a pairing effect smaller than this cannot be told apart from initialisation noise.")
if int((d_hp1 < 0).sum()) == len(d_hp1):
    print("   -> in EVERY seed the random-pair control scores higher than the real pairs: HP1 is not supported and points the other way. Both stay below the untrained embedding,")
    print("      and HP1-Random embeddings are nearly a random projection (mean_cos ~ 1), so this is not evidence that random pairs carry meaning.")
elif int((d_hp1 > 0).sum()) == len(d_hp1):
    print("   -> real pairs score higher in every seed (compare the size of the gap with the seed SD above and with the untrained baseline before calling it support for HP1).")
else:
    print("   -> the sign of the difference changes between seeds: no consistent HP1 effect.")
print(f"3. SELECTION RULE across seeds: MaxSpread - Primary = {d_sel.mean():+.4f} +/- {d_sel.std():.4f}, positive in {int((d_sel > 0).sum())}/{len(d_sel)} seeds.")
print("4. SCALE SWEEP (trained head minus untrained embedding of the same scale, mean over seeds):")
for r in sagg.itertuples():
    print(f"   {r.model:<18} {r.scale:<5} untrained AP {r.raw_AP:.3f} [{r.raw_lo:.3f},{r.raw_hi:.3f}] | trained AP {r.AP_mean:.3f} +/- {r.AP_sd:.3f} | dAP {r.dAP_mean:+.3f} | below untrained in {r.seeds_below_raw}/{r.n} seeds")
raw_aps = [raw_by_scale[s].mean() for s in SCALES]
print(f"   Untrained-embedding AP by scale: " + ", ".join(f"{s} {v:.3f}" for s, v in zip(SCALES, raw_aps)) + f" (spread {max(raw_aps) - min(raw_aps):.3f}; chance {CHANCE[(P, 'MC')]['chance_AP']:.3f}).")
print("   Four scales are too few for a correlation; read the table, not a coefficient. 'No gain' is not 'scale-invariant'.")
r_ = ragg.set_index("model")
print(f"5. REVERSE DIRECTION (fold 1 -> fold 0, {N_SEEDS_REVERSE} seeds; untrained AP {rev_base['RAW 650M']['AP'].mean():.3f}, chance {chr_['chance_AP']:.3f}):")
for m, r in r_.iterrows():
    print(f"   {m:<22} AP {r['AP_mean']:.3f} +/- {r['AP_sd']:.3f}; dAP vs untrained {r['dAP_mean']:+.3f}; below untrained in {int(r['seeds_below_raw'])}/{int(r['n'])} seeds")
def _status(r):
    if int(r["seeds_below_raw"]) == int(r["n"]) and r["dAP_mean"] < -2 * r["dAP_sd"]: return "below the untrained embedding in every seed"
    if abs(r["dAP_mean"]) <= 2 * max(r["dAP_sd"], 1e-9): return "indistinguishable from the untrained embedding (|dAP| <= 2 SD) - not evidence of equivalence"
    return "mixed"
for m, r in r_.iterrows(): print(f"   {m:<22} -> {_status(r)}")
print(f"\n   total wall time {time.time() - T_START:.0f}s")
meta = {"s2_fingerprint": S2_FP, "seeds": SEEDS, "n_seeds_scale": N_SEEDS_SCALE, "n_seeds_reverse": N_SEEDS_REVERSE, "versions": VERSIONS, "date": str(datetime.date.today())}
json.dump(meta, open(f"{RES}/step6_meta.json", "w"), indent=2)
print("=" * 80)

---
## Outputs

| File | Contents |
|---|---|
| `step6_seed_runs.csv`, `step6_seed_spread.csv` | every run and the mean / SD / range per recipe (650M) |
| `step6_scale_runs.csv`, `step6_scale_sweep.csv` | 35M / 150M / 650M / 3B: untrained AP, trained AP, dAP per scale |
| `step6_reverse_runs.csv`, `step6_reverse.csv` | fold 1 → fold 0 |
| `step6_meta.json` | S2 fingerprint, seeds, package versions |

**Next:** Step 7 packages the benchmark, the baseline scorer and the integration contract.